# Additional experiments

Mirrors `README.md`'s run order; each stage checkpoints, so a cell can be rerun to resume. See `PROVENANCE.md` before citing numbers.

In [1]:
# one-time dependencies beyond the repo dev environment
%pip -q install lightgbm pmlb rdata

Note: you may need to restart the kernel to use updated packages.


In [2]:
# datasets from canonical sources (network access required)
!python fetch_data.py --all

== communities ==
  fetching https://archive.ics.uci.edu/ml/machine-learning-databases/communities/communities.data
  checksum communities_crime_numeric.csv: OK
== superconduct ==
  fetching https://archive.ics.uci.edu/ml/machine-learning-databases/00464/superconduct.zip
  checksum superconductivity.csv: MISMATCH (recorded d20a399d3ce3026f205e15ecc7301531, got ae79c46378c122f782e502628f865f73)
== tecator ==
  fetching https://raw.githubusercontent.com/gogorazet/tecator/main/csvtecator.csv
  checksum tecator.csv: MISMATCH (recorded cd4a3ebe9e8cdf3c83a9ad2bc16f477e, got 36e26177a538555a980803e1ce6ea60a)
== eyedata ==
  fetching https://codeload.github.com/cran/flare/tar.gz/refs/heads/master
  FAILED: FileNotFoundError: [Errno 2] No such file or directory: '/home/dara/.local/share/Trash/files/additional_experiments/data/_data/eyedata.rda'
== riboflavin ==
  fetching https://codeload.github.com/cran/hdi/tar.gz/refs/heads/master
  checksum riboflavin.csv: OK
== ct_slices ==
  fetching https

## Comparison study

In [3]:
!python bench.py highdim ridge_raw,rf_raw,lgbm_raw,beamfeat,beamfeat_ridge 5 results/highdim_fast.json

communities_p100 ridge_raw     s0 R2=+0.6654     1.6s n_new=None
communities_p100 rf_raw        s0 R2=+0.6597     3.4s n_new=None
communities_p100 lgbm_raw      s0 R2=+0.6614     3.3s n_new=None
communities_p100 beamfeat      s0 R2=+0.6558    11.5s n_new=10
communities_p100 beamfeat_ridge s0 R2=+0.6563    10.8s n_new=10
communities_p100 ridge_raw     s1 R2=+0.5970     1.6s n_new=None
communities_p100 rf_raw        s1 R2=+0.5965     3.4s n_new=None
communities_p100 lgbm_raw      s1 R2=+0.5695     3.2s n_new=None
communities_p100 beamfeat      s1 R2=+0.6067    11.0s n_new=10
communities_p100 beamfeat_ridge s1 R2=+0.6061    10.7s n_new=10
communities_p100 ridge_raw     s2 R2=+0.6742     1.6s n_new=None
communities_p100 rf_raw        s2 R2=+0.6600     3.3s n_new=None
communities_p100 lgbm_raw      s2 R2=+0.6509     3.3s n_new=None
communities_p100 beamfeat      s2 R2=+0.6471    10.4s n_new=9
communities_p100 beamfeat_ridge s2 R2=+0.6477    10.5s n_new=9
communities_p100 ridge_raw     s3 R2

In [ ]:
!python bench.py highdim featuretools,openfe 5 results/highdim_constructors.json

communities_p100 featuretools  s0 R2=+0.6615    17.4s n_new=14850
  0%|                                                    | 0/4 [00:00<?, ?it/s][LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000030 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 24
[LightGBM] [Info] Number of data points in the train set: 149, number of used features: 1
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po

`autofeat` runs from its pinned venv as in the original study (`../independent/setup_env.sh`), then:

    python bench.py highdim autofeat 5 results/highdim_autofeat.json

## Search characterisation

In [ ]:
!python depth_ladder.py --seeds 20 --out results/depth_ladder.json

In [ ]:
!python depth_ladder.py --seeds 20 --binary mul,div --out results/depth_ladder_ops_muldiv.json
!python depth_ladder.py --seeds 20 --unary square,abs --out results/depth_ladder_ops_nosqrt.json

In [ ]:
!python scalability.py --p-grid 10,30,100,300,1000 --seeds 5 --out results/scalability.json

## Selector comparison

In [ ]:
!python selector_comparison.py --trials 100 --out results/selector_comparison.json
!python selector_comparison.py --trials 100 --m 100 --k 10 --out results/selector_comparison_m100.json

## Split stability

In [ ]:
!python split_stability.py --splits 30 --out results/split_stability.json

In [ ]:
!python multisplit.py data/tecator.csv fat --splits 20

## Tables and figures

In [ ]:
import json, glob, collections
import numpy as np, pandas as pd

rows = []
for f in glob.glob("results/highdim_*.json"):
    rows += json.load(open(f))
df = pd.DataFrame(rows)
ok = df[df.error.isna()]
summary = (ok.groupby("method")
             .agg(mean_r2=("r2", "mean"), worst=("r2", "min"),
                  neg=("r2", lambda s: int((s < 0).sum())),
                  mean_features=("n_new", "mean"), mean_s=("seconds", "mean"))
             .sort_values("mean_r2", ascending=False).round(3))
errors = (df[df.error.notna()].groupby("method")
            .error.apply(lambda s: collections.Counter(e.split(":")[0] for e in s)))
print(summary, "\n\nrecorded failures and budget hits:\n", errors, sep="")

piv = ok.pivot_table(index=["dataset", "split"], columns="method", values="r2")
print("\nper-dataset means:\n", piv.groupby("dataset").mean().round(3))

try:
    st = json.load(open("results/split_stability.json"))
    for k, v in st.items():
        print(f"{k:14s} R2 {v['r2_mean']:.3f}\u00b1{v['r2_std']:.3f} "
              f"Jaccard(eq) {v['jaccard_mean']:.2f} stable classes {len(v['stable_features'])}")
except FileNotFoundError:
    pass

In [ ]:
!python make_figures.py